# Landfill Waste Sorting with Deep Learning

**IT8414 Deep Learning for Data Analytics · Group 3 · Bahrain Polytechnic, Semester 1, 2026–2027**

Members: Sayed Fadhul Hashem, Ali Haji, Mohamed Alradhi  
Tutor: Dr. Shomona Jacob

This notebook classifies photos of landfill waste into nine material classes using the RealWaste dataset (see `docs/DATA.md`).

Each section opens with a short note on what the code does and why.

## 1.0 Setup

This section loads the libraries and sets the values the next sections use: the random seed, the data paths and the nine class names. The fixed seed makes every random step give the same result on every run.

In [ ]:
# Setup for sections 2.0 to 4.0: imports, seed, data paths and class names
import hashlib, shutil, sys, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42                                  # fixed seed: the same photo sample and split on every run

IN_COLAB = "google.colab" in sys.modules   # True on Google Colab, False on our laptops
# Colab works in /content, so the data goes to /content/data; a laptop uses the repo's data/ folder
DATA_DIR = Path("data") if IN_COLAB else Path("../data")
ZIP_PATH = DATA_DIR / "realwaste.zip"
IMAGE_DIR = DATA_DIR / "realwaste-main" / "RealWaste"    # one folder of photos per class
DATA_URL = "https://archive.ics.uci.edu/static/public/908/realwaste.zip"
# SHA-256 fingerprint of the zip we documented in docs/DATA.md; 2.0 stops if the file is different
ZIP_SHA256 = "1ede08b32358ee62065bcc1c8cb47a2ece04e8dff5b1fb53342bb8750895b2f3"

# Alphabetical, like the folders: a class's position in this list is its label (0-8)
CLASS_NAMES = ["Cardboard", "Food Organics", "Glass", "Metal", "Miscellaneous Trash",
               "Paper", "Plastic", "Textile Trash", "Vegetation"]

## 2.0 Data loading

This section puts the RealWaste images on the machine and turns the class folders into a table with one row per image: file path, file name, class name and number label. The download runs only once and is checked against the fingerprint in `docs/DATA.md`, so every run uses exactly the same files. We recount the images per class before any analysis, because a wrong count would change every result that follows.

In [ ]:
# Download and unzip RealWaste once; later runs reuse the files
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)       # 656.6 MB, takes a few minutes

# Stop if the zip is not exactly the file we documented (a new version or a broken download)
with open(ZIP_PATH, "rb") as f:
    digest = hashlib.file_digest(f, "sha256").hexdigest()
assert digest == ZIP_SHA256, "Zip differs from docs/DATA.md: delete it and run again"

if not IMAGE_DIR.exists():
    shutil.unpack_archive(ZIP_PATH, DATA_DIR)            # creates realwaste-main/RealWaste/<class>/
print("Images ready in:", IMAGE_DIR)

In [ ]:
# Image table: one row per photo with its path, file name, class name and label
rows = []
for label, name in enumerate(CLASS_NAMES):                   # label = the class's position (0-8)
    for path in sorted((IMAGE_DIR / name).glob("*.jpg")):    # sorted: the same order on every computer
        rows.append({"path": str(path), "file": path.name, "class_name": name, "label": label})
df = pd.DataFrame(rows)

# Check nothing is missing: our counts must equal the counts published with the dataset (docs/DATA.md)
PUBLISHED = {"Cardboard": 461, "Food Organics": 411, "Glass": 420, "Metal": 790,
             "Miscellaneous Trash": 495, "Paper": 500, "Plastic": 921,
             "Textile Trash": 318, "Vegetation": 436}
print("Total images:", len(df))
print("Every class matches the published count:", df["class_name"].value_counts().to_dict() == PUBLISHED)
df.head()

## 3.0 Exploratory data analysis

This section looks at the photos before the split. Figure 1 checks how balanced the classes are, Figure 2 checks for private information, and Figure 3 looks for photos of the same item taken twice. The figures only describe the dataset and change nothing in it, so looking at every photo here leaks no information between the sets made in section 4.0.

In [ ]:
# Figure 1: images per class, to see how unbalanced the classes are
counts = df["class_name"].value_counts().sort_values()    # smallest first, so the largest bar is on top

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(counts.index, counts.values)
for i, n in enumerate(counts):                            # write each bar's count and share at its end
    ax.text(n, i, f" {n} ({n / len(df):.1%})", va="center", fontsize=8)
ax.set_xlim(0, counts.max() * 1.25)                       # room for the labels
ax.set_xlabel("Number of images")
ax.set_title(f"Figure 1: Images per class ({len(df):,} photos)")
plt.tight_layout(); plt.show()

print(f"Largest / smallest class: {counts.max() / counts.min():.1f} to 1")
print(f"Always guessing '{counts.idxmax()}' would score {counts.max() / len(df):.1%} accuracy")

**What Figure 1 shows and what we decide from it.** Plastic has 921 photos and Textile Trash only 318, a gap of 2.9 to 1, and always guessing Plastic would already score 19.4% accuracy. Decision: section 4.0 splits the data stratified by class, so training, validation and test keep these same shares.

In [ ]:
# Figure 2: privacy check - can a paper or cardboard photo show a readable name or address?
paper_cardboard = df[df["class_name"].isin(["Cardboard", "Paper"])]
sample = paper_cardboard.groupby("class_name").sample(n=6, random_state=SEED)   # 6 per class, the same every run

fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))        # Cardboard on the top row, Paper below
for ax, row in zip(axes.flat, sample.itertuples()):      # one photo per panel
    ax.imshow(Image.open(row.path))
    ax.set_title(row.file, fontsize=8)
    ax.axis("off")
plt.suptitle("Figure 2: Privacy check on 6 Cardboard (top) and 6 Paper (bottom) photos")
plt.tight_layout(); plt.show()

**What Figure 2 shows.** These 12 photos show packaging with brand names, crumpled paper, a rolled newspaper, a road-safety leaflet, an advertising flyer and a window envelope with a postmark. None shows a readable personal name or address, and at 524 × 524 pixels small print such as body text cannot be read. Envelopes and letters do appear in the Paper class, so the privacy risk is low but not zero.

In [ ]:
# Near-duplicate search: find photos of the same item taken twice, using a 256-bit fingerprint (dHash)
def fingerprint(path):
    """Shrink to 17x16 grey pixels; each bit says whether a pixel is brighter than its left neighbour."""
    grey = np.asarray(Image.open(path).convert("L").resize((17, 16), Image.Resampling.LANCZOS))
    return (grey[:, 1:] > grey[:, :-1]).flatten()        # 16 x 16 = 256 True/False values

prints = np.array([fingerprint(p) for p in df["path"]])  # one row of 256 bits per photo

# Compare each photo with every photo and keep the similar pairs
pairs = []                                               # (bits that differ, photo i, photo j)
for i in range(len(prints)):
    bits = (prints != prints[i]).sum(axis=1)             # how many of the 256 bits differ from photo i
    for j in np.where(bits < 30)[0]:                     # similar photos: fewer than 30 bits differ
        if j > i:                                        # each pair once, never a photo with itself
            pairs.append((bits[j], i, j))
pairs.sort()                                             # closest pairs first

DUP_THRESHOLD = 24   # checked by eye in Figure 3: under 24 bits were re-shots, 26 and above different items
near_dups = [(i, j) for bits, i, j in pairs if bits < DUP_THRESHOLD]
print("Closest pair:", pairs[0][0], "of 256 bits differ (0 would mean two identical photos)")
print(f"Near-duplicate pairs (under {DUP_THRESHOLD} of 256 bits): {len(near_dups)}")

In [ ]:
# Figure 3: the 6 closest pairs - the near-duplicates first, then the closest look-alikes
panels = []                                              # (row in df, title) for each photo shown
for bits, i, j in pairs[:6]:
    verdict = "near-duplicate" if bits < DUP_THRESHOLD else "different items"
    panels += [(i, f"{df.loc[i, 'file']}\n{bits} bits: {verdict}"),
               (j, f"{df.loc[j, 'file']}\n{bits} bits: {verdict}")]

fig, axes = plt.subplots(3, 4, figsize=(12, 9.5))        # two pairs per row, each pair side by side
for ax, (k, title) in zip(axes.flat, panels):
    ax.imshow(Image.open(df.loc[k, "path"]))
    ax.set_title(title, fontsize=8)
    ax.axis("off")
plt.suptitle("Figure 3: Near-duplicate pairs, then the closest pairs that are different items")
plt.tight_layout(); plt.show()

**What Figure 3 shows and what we decide from it.** No two photos are identical: even the closest pair differs in 10 of 256 bits. Only two pairs are near-duplicates, `Food Organics_81/82` (10 bits) and `Metal_216/217` (22 bits), each the same item photographed twice. The four closest pairs above our 24-bit cut-off (26 and 29 bits) are different items on the same concrete floor, so the cut-off sits in the gap between real re-shots and look-alikes. Decision: section 4.0 keeps each near-duplicate pair in the same set, so the test set never holds a copy of a training photo. A fingerprint cannot catch the same item photographed from a very different angle, which is a limitation of this check.

## 4.0 Preprocessing and splitting

This section splits the photos into training (60%), validation (20%) and test (20%) sets, the recommended ratio for datasets under 10,000 samples. The split is stratified, so each set keeps Figure 1's class shares, and grouped, so each near-duplicate pair from Figure 3 stays in one set. A fixed seed makes the split identical on every run, and the checks below prove that nothing leaks between the sets.

In [ ]:
# Stratified, grouped 60/20/20 split: both photos of a near-duplicate pair always land in the same set
df["group"] = df.index                                   # every photo starts in its own group
for i, j in near_dups:
    # photo j's group (and anything already joined to it) moves into photo i's group
    df.loc[df["group"] == df.loc[j, "group"], "group"] = df.loc[i, "group"]

# StratifiedGroupKFold keeps each class's share in every fold and never splits a group
# Test = 1 of 5 folds (20% of all photos)
folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
rest_idx, test_idx = next(folds.split(df, df["label"], df["group"]))
rest = df.iloc[rest_idx]                                 # the other 80%
# Validation = 1 of 4 folds of the rest (20% of all); training = the other 3 folds (60% of all)
folds = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
train_idx, val_idx = next(folds.split(rest, rest["label"], rest["group"]))

df["split"] = ""
df.loc[rest.index[train_idx], "split"] = "train"         # positions in rest -> row labels in df
df.loc[rest.index[val_idx], "split"] = "val"
df.loc[test_idx, "split"] = "test"                       # in df, row labels equal positions (0-4751)
sizes = df["split"].value_counts().loc[["train", "val", "test"]]
print(pd.DataFrame({"images": sizes, "share %": (sizes / len(df) * 100).round(1)}))

In [ ]:
# Checks on the split: near-duplicates kept together, class shares kept, and a fingerprint of the test set
same_set = all(df.loc[i, "split"] == df.loc[j, "split"] for i, j in near_dups)
print("Near-duplicate pairs kept in the same set:", same_set)

# Class shares (%) in each set: stratification should keep them close to the whole dataset
shares = pd.crosstab(df["class_name"], df["split"], normalize="columns") * 100
shares["all"] = df["class_name"].value_counts(normalize=True) * 100
shares = shares[["train", "val", "test", "all"]]
gap = max((shares[s] - shares["all"]).abs().max() for s in ["train", "val", "test"])
print(f"Largest gap between a set's class share and the overall share: {gap:.1f} percentage points")

# Test-set fingerprint: a computer that prints a different value has made a different split
test = df[df["split"] == "test"]
names = sorted(test["class_name"] + "/" + test["file"])
print("Test-set fingerprint:", hashlib.sha256("\n".join(names).encode()).hexdigest()[:12])
shares.round(1)

**What the checks show.** The split gives 2,851 training, 950 validation and 951 test photos (60/20/20). Both near-duplicate pairs stay inside one set, and every class's share in each set is within 0.1 percentage points of its overall share, so the test set is a fair, held-out sample of the whole dataset. If the test-set fingerprint (`e9c3df307b09`) differs on another computer, the split is different there and results cannot be compared.

## 5.0 Baseline model

### 5.1 Architecture

### 5.2 Training

### 5.3 Curves

## 6.0 Improved model

### 6.1 Architecture

### 6.2 Training

### 6.3 Curves

## 7.0 Hyperparameter experiments

## 8.0 Test-set evaluation

## 9.0 Interpretability and error analysis

### 9.1 Grad-CAM

### 9.2 Error analysis

### 9.3 Prediction on any image file

## 10.0 Conclusion